# Benchmark: DLTS-LDS Batched vs Beam Search

Compara la versión optimizada `DLTSLDSBatchedSolver` (inferencia batched por nivel de discrepancia)
contra Beam Search W=32, con distintos valores de `max_disc`.

**Solvers:**
- **LDS-batch-full** (`max_disc=None`): búsqueda completa, inferencia batched.
- **LDS-batch-d5** (`max_disc=5`): igual, corta en discrepancia ≥ 5.
- **LDS-batch-d3** (`max_disc=3`): igual, corta en discrepancia ≥ 3.
- **Beam-W32**: Beam Search W=32 por log-probabilidad acumulada.

Modelos: `v2_actions_rl` (branching) + `v2_cost` (bounding para LDS).

In [1]:
import sys, os, json, copy, time
import torch
import numpy as np

MAIN_SRC  = os.path.abspath('../src')
REPO_SRC  = os.path.abspath('../Repo_Oscar/CPMP-Framework/src')
MODEL_DIR = os.path.abspath('../models')

sys.path.insert(0, MAIN_SRC)

from training.training import load_model
from solvers.dlts_lds_batched import DLTSLDSBatchedSolver
from solvers.beam_search import BeamSearchSolver
from cpmp.layout import read_file
from settings import INSTANCE_FOLDER

print('Main src OK')

for k in [k for k in sys.modules if k == 'generation' or k.startswith('generation.')]:
    del sys.modules[k]
sys.path.insert(0, REPO_SRC)

from models.actions.cpmp_transformer_V2 import CPMPTransformer as OscarCPMPTransformer
from models.cost.cost_predictor_V2 import CostPredictorTransformer
from generation.adapters.input.enriched_layout_adapter import EnrichedLayoutAdapter
from generation.adapters.input.layout.layout_4D_adapter_V2 import Layout4DAdapterV2
from generation.adapters.input.stack_features.stack_features_adapter_V1 import StackFeaturesAdapterV1

print('Repo_Oscar OK')

Main src OK
Repo_Oscar OK


In [2]:
class OscarV2LayoutAdapter:
    def __init__(self, S_max, H_max):
        self._inner = EnrichedLayoutAdapter(
            layout_adapter=Layout4DAdapterV2,
            stack_features_adapter=StackFeaturesAdapterV1,
            S_max=S_max, H_max=H_max
        )
        self.S_max = S_max
    def layout_2_vec(self, layout, H):
        return self._inner.input_2_vec(layout, H)

class OscarV2Wrapper(torch.nn.Module):
    def __init__(self, model, S_max):
        super().__init__()
        self._model = model
        self.S_max  = S_max
        self.hyperparams = model.hyperparams

    def forward(self, L, X, S, H):
        logits_full = self._model(L, X, S, H)
        # S puede ser (N,) cuando se llama en batch — usamos el primer valor
        # (todos los nodos del mismo nivel tienen el mismo S)
        S_int = int(S.flatten()[0].item())
        batch = logits_full.shape[0]
        out   = torch.zeros(batch, S_int * (S_int - 1), device=logits_full.device)
        for src in range(S_int):
            for dst in range(S_int):
                if src == dst:
                    continue
                d_off = dst if dst < src else dst - 1
                oscar_idx  = src * (self.S_max - 1) + d_off
                solver_idx = src * (S_int - 1)      + d_off
                out[:, solver_idx] = logits_full[:, oscar_idx]
        return out

S_MAX, H_MAX = 10, 12
_oscar_base     = load_model(OscarCPMPTransformer, 'v2_actions_rl')
branching_model = OscarV2Wrapper(_oscar_base, S_max=S_MAX)
branching_model.layout_adapter = OscarV2LayoutAdapter(S_max=S_MAX, H_max=H_MAX)
branching_model.eval()

def load_cost_model(name):
    hp = json.load(open(os.path.join(MODEL_DIR, 'hyperparameters', f'{name}.json')))
    m = CostPredictorTransformer(**hp)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'{name}.pth'),
                                  weights_only=True, map_location='cpu'))
    m.eval()
    return m

bounding_model = load_cost_model('v2_cost')

bounding_adapter = EnrichedLayoutAdapter(
    layout_adapter=Layout4DAdapterV2,
    stack_features_adapter=StackFeaturesAdapterV1,
    S_max=S_MAX, H_max=H_MAX
)

hp = branching_model.hyperparams
print(f'Branching: v2_actions_rl  H_dim={hp["H_dim"]}, d_model={hp["d_model"]}')
print(f'Bounding : v2_cost')

Branching: v2_actions_rl  H_dim=12, d_model=64
Bounding : v2_cost


In [3]:
T_LIM = 15.0
W     = 32
SHARED = dict(branching_model=branching_model,
              bounding_model=bounding_model,
              bounding_adapter=bounding_adapter,
              p=0.3, k=3, d=0.8, z=0, time_limit=T_LIM)

solvers = {
    'LDS-batch-full': DLTSLDSBatchedSolver(**SHARED, max_disc=None),
    'LDS-batch-d5'  : DLTSLDSBatchedSolver(**SHARED, max_disc=5),
    'LDS-batch-d3'  : DLTSLDSBatchedSolver(**SHARED, max_disc=3),
    'Beam-W32'      : BeamSearchSolver(branching_model=branching_model,
                                        beam_width=W, expansions_per_state=W,
                                        time_limit=T_LIM),
}

for name, s in solvers.items():
    extra = f'  max_disc={s.max_disc}' if hasattr(s, 'max_disc') else f'  W={s.beam_width}'
    print(f'  {name:<18}: {s.name}{extra}')

  LDS-batch-full    : DLTSLDSBatchedSolver  max_disc=None
  LDS-batch-d5      : DLTSLDSBatchedSolver  max_disc=5
  LDS-batch-d3      : DLTSLDSBatchedSolver  max_disc=3
  Beam-W32          : BeamSearchSolver  W=32


In [4]:
def avg(vals, mask=None):
    data = [v for v, ok in zip(vals, mask or [True]*len(vals)) if ok]
    return sum(data) / len(data) if data else float('nan')

def run_solver(solver, files, H_inf, max_steps):
    solved_list, steps_list, time_list = [], [], []
    for path in files:
        layout = read_file(path, H_inf)
        t0 = time.perf_counter()
        try:
            solved, steps = solver.solve_from_layout(layout, H_inf, max_steps)
        except Exception as e:
            print(f'  ERROR en {os.path.basename(path)}: {e}')
            solved, steps = False, max_steps
        elapsed = time.perf_counter() - t0
        solved_list.append(solved)
        steps_list.append(steps)
        time_list.append(elapsed)
    return solved_list, steps_list, time_list

print('Utilidades definidas.')

Utilidades definidas.


## Benchmark por categorías CVS

In [5]:
CVS_PATH  = INSTANCE_FOLDER / 'benchmarks' / 'CVS'
MAX_STEPS = 100
N_PER_CAT = 20

ALL_CATS = sorted([d for d in os.listdir(CVS_PATH)
                   if (CVS_PATH / d).is_dir() and not d.startswith('10-')])

solver_names = list(solvers.keys())
SEP = '-' * 130

header = f"{'Cat':>6} {'H':>3} {'S':>3}"
for name in solver_names:
    header += f"  {name:>18} {'paso':>5} {'t(s)':>6}"
print(header)
print(SEP)

all_results = {}

for folder_name in ALL_CATS:
    H_r, S_r = [int(x) for x in folder_name.split('-')]
    H_inf = H_r + 2
    folder_path = CVS_PATH / folder_name
    files = sorted([str(folder_path / f)
                    for f in os.listdir(folder_path) if f.endswith('.dat')])[:N_PER_CAT]
    if not files:
        continue

    cat_res = {}
    for name, solver in solvers.items():
        s, st, t = run_solver(solver, files, H_inf, MAX_STEPS)
        cat_res[name] = dict(solved=s, steps=st, time=t)

    all_results[folder_name] = cat_res
    n = len(files)

    row = f"{folder_name:>6} {H_r:>3} {S_r:>3}"
    for name in solver_names:
        r = cat_res[name]
        row += f"  {sum(r['solved']):>3}/{n:<2} {avg(r['steps'], r['solved']):>8.1f} {avg(r['time'], r['solved']):>6.2f}"
    print(row)

print(SEP)

totals = {name: dict(solved=[], steps=[], time=[]) for name in solver_names}
for cat_res in all_results.values():
    for name in solver_names:
        totals[name]['solved'].extend(cat_res[name]['solved'])
        totals[name]['steps'].extend(cat_res[name]['steps'])
        totals[name]['time'].extend(cat_res[name]['time'])

n_tot = len(totals[solver_names[0]]['solved'])
row = f"{'TOTAL':>6} {'':>3} {'':>3}"
for name in solver_names:
    r = totals[name]
    row += f"  {sum(r['solved']):>3}/{n_tot:<2} {avg(r['steps'], r['solved']):>8.1f} {avg(r['time'], r['solved']):>6.2f}"
print(row)

   Cat   H   S      LDS-batch-full  paso   t(s)        LDS-batch-d5  paso   t(s)        LDS-batch-d3  paso   t(s)            Beam-W32  paso   t(s)
----------------------------------------------------------------------------------------------------------------------------------
   3-3   3   3   20/20     10.3   0.06   20/20     10.3   0.06   20/20     10.3   0.06   20/20      9.8   0.52
   3-4   3   4   20/20      8.8   0.06   20/20      8.8   0.05   20/20      8.8   0.05   20/20      8.8   0.54
   3-5   3   5   20/20     10.6   0.11   20/20     10.6   0.10   20/20     10.6   0.08   20/20     10.4   0.70
   3-6   3   6   20/20     11.8   0.29   20/20     11.8   0.19   20/20     11.8   0.12   20/20     11.7   0.88
   3-7   3   7   20/20     12.9   0.71   20/20     12.9   0.28   20/20     13.0   0.14   20/20     13.1   1.09
   3-8   3   8   20/20     13.2   2.48   20/20     13.2   0.46   20/20     13.2   0.18   20/20     13.2   1.31
   4-4   4   4   20/20     16.8   0.35   20/20     16.9 

KeyboardInterrupt: 

## Resumen: calidad y speedup

In [ ]:
ref_name = 'LDS-batch-full'
ref = totals[ref_name]

mask_all = [all(totals[n]['solved'][i] for n in solver_names)
            for i in range(n_tot)]
n_common = sum(mask_all)

print(f'Instancias donde todos los solvers resolvieron: {n_common}/{n_tot}')
print()
print(f'{"Solver":<20} {"Avg pasos":>10} {"vs LDS-full":>12} {"Avg t(s)":>10} {"Speedup":>8}')
print('-' * 65)

ref_steps = [s for s, ok in zip(ref['steps'], mask_all) if ok]
ref_time  = avg(ref['time'], ref['solved'])

for name in solver_names:
    r = totals[name]
    steps_common = [s for s, ok in zip(r['steps'], mask_all) if ok]
    avg_steps = np.mean(steps_common)
    avg_t     = avg(r['time'], r['solved'])
    vs_ref    = (avg_steps / np.mean(ref_steps) - 1) * 100
    speedup   = ref_time / avg_t if avg_t > 0 else float('nan')
    marker    = '  <- ref' if name == ref_name else ''
    print(f'{name:<20} {avg_steps:>10.2f} {vs_ref:>+11.1f}% {avg_t:>10.3f}  {speedup:>7.2f}x{marker}')

print()
print(f'Comparacion por pares vs {ref_name} (instancias comunes):')
print(f'{"":20} {"mejor":>8} {"igual":>8} {"peor":>8}')
for name in solver_names:
    if name == ref_name:
        continue
    r = totals[name]
    pairs = [(s1, s2) for s1, s2, ok in
             zip(r['steps'], ref['steps'], mask_all) if ok]
    better = sum(1 for a, b in pairs if a < b)
    equal  = sum(1 for a, b in pairs if a == b)
    worse  = sum(1 for a, b in pairs if a > b)
    print(f'  {name:<18}: {better:>5} ({better/n_common:>5.1%})  '
          f'{equal:>5} ({equal/n_common:>5.1%})  {worse:>5} ({worse/n_common:>5.1%})')

## Visualización

In [ ]:
import matplotlib.pyplot as plt

cats_sorted = sorted(all_results.keys())
colors = ['#ed7d31', '#70ad47', '#ffc000', '#5b9bd5']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Pasos promedio por categoría ──────────────────────────────────────────
ax = axes[0]
x = np.arange(len(cats_sorted))
width = 0.8 / len(solver_names)
offsets = np.linspace(-(len(solver_names)-1)/2, (len(solver_names)-1)/2, len(solver_names)) * width

for i, (name, color) in enumerate(zip(solver_names, colors)):
    avg_steps = [avg(all_results[c][name]['steps'], all_results[c][name]['solved'])
                 for c in cats_sorted]
    ax.bar(x + offsets[i], avg_steps, width, label=name, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(cats_sorted, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Avg pasos')
ax.set_title('Pasos promedio por categoria')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# ── 2. Tiempo promedio por categoría ─────────────────────────────────────────
ax2 = axes[1]
for i, (name, color) in enumerate(zip(solver_names, colors)):
    avg_times = [avg(all_results[c][name]['time'], all_results[c][name]['solved'])
                 for c in cats_sorted]
    ax2.bar(x + offsets[i], avg_times, width, label=name, color=color, alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels(cats_sorted, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Avg tiempo (s)')
ax2.set_title('Tiempo promedio por categoria')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

# ── 3. Scatter: LDS-batch-full vs Beam-W32 (pasos por instancia) ─────────────
ax3 = axes[2]
beam_name = 'Beam-W32'
pairs_ok = [(s_lds, s_beam)
            for s_lds, ok_lds, s_beam, ok_beam
            in zip(totals[ref_name]['steps'], totals[ref_name]['solved'],
                   totals[beam_name]['steps'], totals[beam_name]['solved'])
            if ok_lds and ok_beam]

if pairs_ok:
    xs, ys = zip(*pairs_ok)
    ax3.scatter(xs, ys, alpha=0.4, s=18, color='#7030a0')
    mx = max(max(xs), max(ys)) * 1.05
    ax3.plot([0, mx], [0, mx], 'r--', lw=1.5, label='igual')
    ax3.set_xlabel(f'Pasos {beam_name}')
    ax3.set_ylabel(f'Pasos {ref_name}')
    ax3.set_title(f'{ref_name} vs {beam_name}\n(bajo diagonal = LDS mejor)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Análisis del efecto de max_disc

In [ ]:
batch_solvers = [n for n in solver_names if 'batch' in n]

print(f'Trade-off calidad vs tiempo (instancias comunes: {n_common})')
print()
print(f'{"Solver":<20} {"Avg pasos":>10} {"delta pasos":>12} {"Avg t(s)":>10} {"Speedup vs full":>16}')
print('-' * 72)

ref_steps_arr = np.array([s for s, ok in zip(ref['steps'], mask_all) if ok])
ref_avg_time  = avg(ref['time'], ref['solved'])

print(f'{ref_name:<20} {np.mean(ref_steps_arr):>10.2f} {"":>12} {ref_avg_time:>10.3f}  {"1.00x":>14}')

for name in batch_solvers:
    if name == ref_name:
        continue
    r = totals[name]
    steps_common = np.array([s for s, ok in zip(r['steps'], mask_all) if ok])
    avg_steps = np.mean(steps_common)
    avg_t     = avg(r['time'], r['solved'])
    delta     = avg_steps - np.mean(ref_steps_arr)
    speedup   = ref_avg_time / avg_t if avg_t > 0 else float('nan')
    print(f'{name:<20} {avg_steps:>10.2f} {delta:>+12.2f} {avg_t:>10.3f}  {speedup:>13.2f}x')